In [1]:
!pip install -q segmentation-models-pytorch 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.6 MB/s eta 0:00:00


In [2]:
import os
import json
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import albumentations as A
import segmentation_models_pytorch as smp
from albumentations.pytorch import ToTensorV2

In [9]:
# ==========================================
# 1. КОНФИГУРАЦИЯ ДЛЯ ПСЕВДО-ЛЕЙБЕЛИНГА
# ==========================================
UNLABELED_DIR = Path(r"/kaggle/input/competitions/dl-lab-3-product-segmentation/unlabeled/images")

# Папки, куда мы сохраним идеальные предсказания
PSEUDO_OUT_DIR = Path("/kaggle/working/pseudo_data")
PSEUDO_IMAGES_DIR = PSEUDO_OUT_DIR / "images"
PSEUDO_MASKS_DIR = PSEUDO_OUT_DIR / "masks"

# Создаем структуру папок
PSEUDO_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
PSEUDO_MASKS_DIR.mkdir(parents=True, exist_ok=True)

# ПУТИ К ВЕСАМ (Проверь правильность путей!)
WEIGHTS_DIR_CONF1 = Path(r"/kaggle/input/datasets/tkachenko1van/conf1-unetpp-effb4")
WEIGHTS_DIR_CONF2 = Path(r"/kaggle/input/datasets/tkachenko1van/conf2-manet-mitb3")
WEIGHTS_DIR_CONF3 = Path(r"/kaggle/input/datasets/tkachenko1van/conf3-deeplab-convnext") 

IMG_SIZE = 320
THRESHOLD = 0.3936
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [4]:
# ==========================================
# 2. ЗАГРУЗКА МОДЕЛЕЙ (С идеальными OOF весами)
# ==========================================
models_with_weights = []
print("🚀 Загрузка моделей в память...")

for fold in range(1, 6):
    p = WEIGHTS_DIR_CONF1 / f"conf1_unetpp_effb4_fold{fold}.pth"
    if p.exists():
        m = smp.UnetPlusPlus(encoder_name="efficientnet-b4", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models_with_weights.append((m, 0.1891))

for fold in range(1, 6):
    p = WEIGHTS_DIR_CONF2 / f"conf2_manet_mitb3_fold{fold}.pth"
    if p.exists():
        m = smp.MAnet(encoder_name="mit_b3", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models_with_weights.append((m, 0.4435))

for fold in range(1, 6):
    p = WEIGHTS_DIR_CONF3 / f"conf3_deeplab_convnext_fold{fold}.pth"
    if p.exists():
        m = smp.DeepLabV3Plus(encoder_name="tu-convnext_small", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models_with_weights.append((m, 0.3673))
        
print(f"✅ Всего загружено моделей в ансамбль: {len(models_with_weights)}")

🚀 Загрузка моделей в память...
✅ Всего загружено моделей в ансамбль: 15


In [5]:
# ==========================================
# 3. ПОСТПРОЦЕССИНГ
# ==========================================
def postprocess_mask(mask: np.ndarray) -> np.ndarray:
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = (labels == largest_label).astype(np.uint8)
    else:
        return np.zeros_like(mask)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cv2.drawContours(mask, contours, -1, 1, thickness=cv2.FILLED)
        
    return mask

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.tolist(), separators=(",", ":"))

In [6]:
# ==========================================
# 4. ФУНКЦИЯ ПРЕДСКАЗАНИЯ (Адаптирована для PL)
# ==========================================
def predict_for_pseudo_labeling(image_bgr: np.ndarray, models_with_weights: list, threshold=THRESHOLD):
    original_h, original_w = image_bgr.shape[:2]
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    augmented = val_transform(image=image_rgb)
    img_orig = augmented['image'].unsqueeze(0).to(DEVICE)

    img_h  = torch.flip(img_orig, dims=[3])
    img_v  = torch.flip(img_orig, dims=[2])
    img_hv = torch.flip(img_orig, dims=[2, 3])

    ensemble_probs = torch.zeros((1, 1, IMG_SIZE, IMG_SIZE), device=DEVICE)
    total_weight = 0.0
    
    for model, weight in models_with_weights:
        with torch.amp.autocast('cuda'):
            prob_orig = torch.sigmoid(model(img_orig))
            prob_h = torch.flip(torch.sigmoid(model(img_h)), dims=[3])
            prob_v = torch.flip(torch.sigmoid(model(img_v)), dims=[2])
            prob_hv = torch.flip(torch.sigmoid(model(img_hv)), dims=[2, 3])
            
            model_probs = (prob_orig + prob_h + prob_v + prob_hv)
            ensemble_probs += (model_probs * weight)
            total_weight += (4.0 * weight)
            
    ensemble_probs /= total_weight
    probs_np = ensemble_probs[0, 0].cpu().numpy().astype(np.float32)
    
    if probs_np.shape != (original_h, original_w):
        probs_np = cv2.resize(probs_np, (original_w, original_h), interpolation=cv2.INTER_LINEAR)
        
    # --- СТАТИСТИКА ДЛЯ ФИЛЬТРАЦИИ ---
    # 1. Считаем пиксели в зоне неуверенности (от 0.3 до 0.7)
    uncertain_pixels = np.sum((probs_np > 0.3) & (probs_np < 0.7))
    total_pixels = original_h * original_w
    uncertainty_ratio = uncertain_pixels / total_pixels
    
    # 2. Бинаризация и детектор галлюцинаций
    raw_binary_mask = (probs_np > threshold).astype(np.uint8)
    final_mask = postprocess_mask(raw_binary_mask)
    
    # Сравниваем маску ДО и ПОСЛЕ удаления мусора (LCC)
    raw_area = np.sum(raw_binary_mask)
    final_area = np.sum(final_mask)
    removed_area_ratio = (raw_area - final_area) / (raw_area + 1e-6) # Сколько % маски было выброшено
    
    return final_mask, uncertainty_ratio, removed_area_ratio

In [10]:
# ==========================================
# 5. ГЕНЕРАЦИЯ ПСЕВДО-ЛЕЙБЛОВ И ОТБОР
# ==========================================
unlabeled_paths = sorted(list(UNLABELED_DIR.glob("*.jpg"))) + sorted(list(UNLABELED_DIR.glob("*.png")))
print(f"📸 Найдено неразмеченных изображений: {len(unlabeled_paths)}")

# НАСТРОЙКИ ЖЕСТКОСТИ ФИЛЬТРА:
MAX_UNCERTAINTY = 0.02  # Максимум 2% пикселей картинки могут быть "сомнительными" (обычно это толщина контура)
MAX_REMOVED_AREA = 0.05 # Максимум 5% маски могло быть удалено при постпроцессинге (защита от галлюцинаций)

accepted_count = 0
rejected_count = 0

with torch.inference_mode():
    for img_path in tqdm(unlabeled_paths, desc="Генерация Pseudo-Labels"):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue

        final_mask, uncert_ratio, removed_ratio = predict_for_pseudo_labeling(img_bgr, models_with_weights)
        
        # ФИЛЬТРАЦИЯ
        is_confident = (uncert_ratio < MAX_UNCERTAINTY)
        is_clean = (removed_ratio < MAX_REMOVED_AREA)
        not_empty = np.sum(final_mask) > 100 # Защита от полностью пустых масок
        
        if is_confident and is_clean and not_empty:
            # Идеальная маска! Сохраняем её.
            # 1. Сохраняем оригинальное изображение
            dest_img_path = PSEUDO_IMAGES_DIR / img_path.name
            cv2.imwrite(str(dest_img_path), img_bgr)
            
            # 2. Сохраняем маску (Умножаем на 255, чтобы белое было белым!)
            mask_name = img_path.stem + ".png" # Маски всегда в .png
            dest_mask_path = PSEUDO_MASKS_DIR / mask_name
            cv2.imwrite(str(dest_mask_path), final_mask * 255)
            
            accepted_count += 1
        else:
            rejected_count += 1

print(f"\n🎯 РЕЗУЛЬТАТЫ ОТБОРА:")
print(f"✅ Принято идеальных масок: {accepted_count}")
print(f"❌ Отбраковано (неуверенных): {rejected_count}")

📸 Найдено неразмеченных изображений: 350


Генерация Pseudo-Labels:  83%|████████▎ | 291/350 [07:15<01:26,  1.46s/it]/tmp/ipykernel_55/1890574854.py:48: RuntimeWarning: overflow encountered in scalar subtract
  removed_area_ratio = (raw_area - final_area) / (raw_area + 1e-6) # Сколько % маски было выброшено
Генерация Pseudo-Labels: 100%|██████████| 350/350 [08:42<00:00,  1.49s/it]


🎯 РЕЗУЛЬТАТЫ ОТБОРА:
✅ Принято идеальных масок: 284
❌ Отбраковано (неуверенных): 66


In [11]:
# ==========================================
# 6. СОЗДАНИЕ ZIP-АРХИВА ДЛЯ СКАЧИВАНИЯ
# ==========================================
print("\n📦 Упаковка в ZIP-архив...")
zip_filename = "pseudo_labels_confident"
shutil.make_archive(zip_filename, 'zip', PSEUDO_OUT_DIR)

print(f"🎉 Готово! Архив {zip_filename}.zip сохранен в /kaggle/working/")


📦 Упаковка в ZIP-архив...
🎉 Готово! Архив pseudo_labels_confident.zip сохранен в /kaggle/working/
